# MultiBench Part 1: MOSI — Early Fusion / Supervised Learning

This notebook demonstrates basic usage of MultiBench with the MOSI affective computing dataset using a simple ConcatEarly fusion model and supervised learning training loop.

It is designed to run on **Google Colab**.

In [ ]:
# Mount Google Drive directly to /content/ so MyDrive is at /content/MyDrive/
from google.colab import drive
drive.mount("/content/")

## Clone or update MultiBench

The repo will be cloned into your Google Drive under . If it already exists, we pull the latest changes instead.

In [ ]:
import os

repo_dir = "/content/MyDrive/MultiBench"

if os.path.isdir(os.path.join(repo_dir, ".git")):
    print("Repo already exists, pulling latest changes...")
    !git -C "$repo_dir" pull
else:
    print("Cloning MultiBench...")
    !git clone https://github.com/human-ai-lab/MultiBench.git "$repo_dir"

print(f"Repo location: {repo_dir}")

## Setup Python path

Add the repo root to  so all MultiBench imports resolve correctly.

In [ ]:
import sys

if repo_dir not in sys.path:
    sys.path.insert(0, repo_dir)

print(f"Using MultiBench from: {repo_dir}")

## Check and download MOSI data

Data is stored at `MultiBench/data/affect/mosi_raw.pkl` — inside the repo on Drive, so it persists across sessions and matches the path used by the local `.py` scripts.

In [ ]:
data_dir  = os.path.join(repo_dir, "data", "affect")
data_path = os.path.join(data_dir, "mosi_raw.pkl")

os.makedirs(data_dir, exist_ok=True)

if os.path.exists(data_path):
    print(f"Data found: {data_path}")
else:
    print("Downloading mosi_raw.pkl (one-time)...")
    !wget -q --show-progress -O "$data_path" https://filedn.eu/lDTxyzlMbdMJJq0AvECx20X/mosi_raw.pkl
    print(f"Saved to: {data_path}")

print(f"data_path = {data_path}")

## Install dependencies

In [ ]:
!pip install -q memory-profiler

## Setup device

In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

**Note about torchtext compatibility:**

This notebook uses GloVe embeddings for text processing. MultiBench includes its own GloVe loader () that automatically replaces torchtext when needed. The first run will download GloVe embeddings (~2 GB) and cache them for future use.

## Load MOSI dataset

In [ ]:
from datasets.affect.get_data import get_dataloader

traindata, validdata, testdata = get_dataloader(
    data_path, robust_test=False, max_pad=True, data_type="mosi", max_seq_len=50)

## Define encoders

We use Identity encoders to pass raw modality features through unchanged. MOSI has three modalities: text, audio, and video.

In [ ]:
from unimodals.common_models import GRU, MLP, Sequential, Identity

encoders = [Identity().to(device), Identity().to(device), Identity().to(device)]

## Define fusion module

We use ConcatEarly, which concatenates all modality inputs along the feature dimension before any processing.

In [ ]:
from fusions.common_fusions import ConcatEarly  # noqa

fusion = ConcatEarly().to(device)

## Define prediction head

In [ ]:
head = Sequential(
    GRU(409, 512, dropout=True, has_padding=False, batch_first=True, last_only=True),
    MLP(512, 512, 1)
).to(device)

## Train

In [ ]:
from training_structures.Supervised_Learning import train, test
# train on 5 epoch, increase this for better performance 
train(encoders, fusion, head, traindata, validdata, 5,
      task="regression", optimtype=torch.optim.AdamW,
      is_packed=False, lr=1e-3, save="mosi_ef_r0.pt",
      weight_decay=0.01, objective=torch.nn.L1Loss())

## Test

In [ ]:
print("Testing:")
model = torch.load("mosi_ef_r0.pt", map_location=device, weights_only=False).to(device)
test(model, testdata, "affect", is_packed=False,
     criterion=torch.nn.L1Loss(), task="posneg-classification", no_robust=True)

That concludes Part 1! Feel free to open an issue on GitHub if you have any questions.